# 01 · Preparación de datos

Descarga el dataset desde el repositorio UCI, separa entrenamiento y prueba (80/20, estratificado) **antes** de cualquier transformación y ajusta la codificación y el escalado solo con el entrenamiento, para evitar fuga de información.

Salidas:
- `outputs/splits/`: datos en escala original (los usan CTGAN, TVAE y las métricas de fidelidad).
- `outputs/data/`: datos escalados para los clasificadores, y los objetos de transformación.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from ucimlrepo import fetch_ucirepo

import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)
from utils.config_loader import load_config

config = load_config("dataset_config.yaml")
RANDOM_STATE = config.get("random_state", 42)
print("Dataset:", config["dataset_name"])

## Carga desde UCI

In [ ]:
dataset = fetch_ucirepo(id=config["uci_id"])
X = dataset.data.features.copy()
y = dataset.data.targets.copy()

# Algunos datasets de UCI traen espacios al inicio de los nombres de columna
wanted = set(config["numerical_features"]) | set(config["categorical_features"]) | set(config.get("drop_columns", []))
X = X.rename(columns={c: c.strip() for c in X.columns if c not in wanted and c.strip() in wanted})
for col in config.get("drop_columns", []):
    if col in X.columns:
        X = X.drop(columns=col)
y = y[config["target_column"]].squeeze()

num_cols = config["numerical_features"]
cat_cols = config["categorical_features"]
print(f"Registros: {len(X)} | numéricas: {len(num_cols)} | categóricas: {len(cat_cols)}")

## Partición entrenamiento / prueba (80/20, estratificada)

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=1 - config.get("train_size", 0.8),
    stratify=y,
    random_state=RANDOM_STATE,
)

# Codificación de la variable objetivo (ajustada solo con entrenamiento)
if y_train.dtype == object or str(y_train.dtype) == "category":
    target_encoder = LabelEncoder()
    y_train_enc = target_encoder.fit_transform(y_train)
    y_test_enc = target_encoder.transform(y_test)
else:
    target_encoder = None
    y_train_enc = y_train.values
    y_test_enc = y_test.values

## Codificación de variables categóricas y escalado de numéricas

In [ ]:
# Label Encoding de las categóricas (ajustado en entrenamiento, aplicado a prueba)
X_train_enc = X_train_raw.copy()
X_test_enc = X_test_raw.copy()
feature_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X_train_enc[col] = le.fit_transform(X_train_raw[col].astype(str))
    X_test_enc[col] = le.transform(X_test_raw[col].astype(str))
    feature_encoders[col] = le

# Estandarización de las numéricas (solo para los clasificadores y SMOTE-NC)
scaler = StandardScaler()
X_train_scaled = X_train_enc.copy()
X_test_scaled = X_test_enc.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train_enc[num_cols].astype(float))
X_test_scaled[num_cols] = scaler.transform(X_test_enc[num_cols].astype(float))

counter = Counter(y_train_enc)
print(f"Distribución en entrenamiento: { {int(k): int(v) for k, v in counter.items()} }")
print(f"Ratio de desbalance: {max(counter.values()) / min(counter.values()):.2f}")

## Guardado

In [ ]:
SPLIT_DIR = "outputs/splits"
DATA_DIR = "outputs/data"
os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Escala original (codificada)
X_train_enc.to_csv(f"{SPLIT_DIR}/X_train_raw_in.csv", index=False)
X_test_enc.to_csv(f"{SPLIT_DIR}/X_test_raw_in.csv", index=False)
pd.DataFrame({"y": y_train_enc}).to_csv(f"{SPLIT_DIR}/y_train_in.csv", index=False)
pd.DataFrame({"y": y_test_enc}).to_csv(f"{SPLIT_DIR}/y_test_in.csv", index=False)

# Escalados (clasificadores)
X_train_scaled.to_csv(f"{DATA_DIR}/X_train.csv", index=False)
X_test_scaled.to_csv(f"{DATA_DIR}/X_test.csv", index=False)
pd.DataFrame({"y": y_train_enc}).to_csv(f"{DATA_DIR}/y_train.csv", index=False)
pd.DataFrame({"y": y_test_enc}).to_csv(f"{DATA_DIR}/y_test.csv", index=False)

joblib.dump(feature_encoders, f"{DATA_DIR}/feature_encoders.pkl")
joblib.dump(target_encoder, f"{DATA_DIR}/target_encoder.pkl")
joblib.dump(scaler, f"{DATA_DIR}/scaler.pkl")
print("Preparación completada.")